In [1]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
project = os.getenv('GOOGLE_CLOUD_PROJECT')


# model
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash-lite",
    vertexai = True,
    project = project
)

In [3]:
# lets define basic tools
from langchain_core.tools import tool

# create and register tools

@tool
def add(a: int|float, b: int|float) -> int|float:
    """Adds two numbers

    Args:
        a (int | float): first argument
        b (int | float): second argument

    Returns:
        int|float: a + b
    """
    return a + b


In [4]:
# Make llm aware of this tools
llm_with_tools = llm.bind_tools([add])

In [5]:
question = "What is the result of 4 + 5 ?"

In [6]:
response_without_tools = llm.invoke(question)

In [8]:
# process response
response_without_tools.pretty_print()

================================== Ai Message ==================================

The result of 4 + 5 is **9**.


In [7]:
response_with_tools = llm_with_tools.invoke(question)

In [9]:
# process response
response_with_tools.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  add (c0d9c852-2b1e-4d27-96d3-e653837c3d16)
 Call ID: c0d9c852-2b1e-4d27-96d3-e653837c3d16
  Args:
    a: 4
    b: 5


In [10]:
# Now lets add more tools
@tool
def subtract(a: int | float, b: int | float) -> int | float:
    """Subtracts second number from first number

    Args:
        a (int | float): first argument
        b (int | float): second argument

    Returns:
        int | float: a - b
    """
    return a - b


@tool
def multiply(a: int | float, b: int | float) -> int | float:
    """Multiplies two numbers

    Args:
        a (int | float): first argument
        b (int | float): second argument

    Returns:
        int | float: a * b
    """
    return a * b


@tool
def divide(a: int | float, b: int | float) -> int | float:
    """Divides first number by second number

    Args:
        a (int | float): first argument
        b (int | float): second argument

    Returns:
        int | float: a / b

    Raises:
        ValueError: If b is 0
    """
    if b == 0:
        raise ValueError("Division by zero is not allowed")
    return a / b


@tool
def modulus(a: int | float, b: int | float) -> int | float:
    """Finds remainder when first number is divided by second number

    Args:
        a (int | float): first argument
        b (int | float): second argument

    Returns:
        int | float: a % b

    Raises:
        ValueError: If b is 0
    """
    if b == 0:
        raise ValueError("Modulus by zero is not allowed")
    return a % b


In [11]:
# lets bind multiple tools to llm

llm_with_tools = llm.bind_tools([add, subtract, multiply, divide, modulus])

In [18]:
question = """
I have purchase a mobile phone at 100000 rupees
I got 15% discount. what i will end up paying
"""

In [19]:
response_with_tools = llm_with_tools.invoke(question)

In [20]:
response_with_tools.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  multiply (7ff70ff2-0c13-4206-8477-53b1a6cf6700)
 Call ID: 7ff70ff2-0c13-4206-8477-53b1a6cf6700
  Args:
    b: 0.15
    a: 100000


In [22]:
response_with_tools.content_blocks

[{'type': 'tool_call',
  'id': '7ff70ff2-0c13-4206-8477-53b1a6cf6700',
  'name': 'multiply',
  'args': {'b': 0.15, 'a': 100000}}]